In [29]:
import polars as pl
from pathlib import Path

In [32]:
def describe(df : pl.DataFrame) -> None:
    """
    Describe the dataframe
    Args:
        df (pl.DataFrame): The dataframe to describe
    returns:
        None
        print: 
            - describe()
            - null values
            - dtypes
    """
    print(df.describe())
    print("\nNumber of Null Values\n", df.null_count())
    print("\nPercentage of Null Values\n",)
    print(((df.null_count() / df.shape[0]) * 100))
    
    print("\nData Types\n", df.dtypes)
    print("\nNumber of duplicated rows\n", df.is_duplicated().sum())


In [33]:
# Get all CSV files from datasets folder
datasets_path = Path("../datasets")
csv_files = list(datasets_path.glob("*.csv"))

# Loop through and read each dataset
datasets = {}
for csv_file in csv_files:
    dataset_name = csv_file.stem  # Get filename without extension
    df = pl.read_csv(csv_file)
    datasets[dataset_name] = df
    
    print(f"\n{'='*60}")
    print(f"Dataset: {dataset_name}")
    print(f"{'='*60}")
    print(f"Shape: {df.shape}")
    print(f"\nColumns: {df.columns}")
    print(f"\nFirst few rows:")
    print(df.head())

    describe(df)



Dataset: workforce_nurses
Shape: (126, 4)

Columns: ['year', 'type', 'sector', 'count']

First few rows:
shape: (5, 4)
┌──────┬───────────────────┬────────────────────────┬───────┐
│ year ┆ type              ┆ sector                 ┆ count │
│ ---  ┆ ---               ┆ ---                    ┆ ---   │
│ i64  ┆ str               ┆ str                    ┆ i64   │
╞══════╪═══════════════════╪════════════════════════╪═══════╡
│ 2006 ┆ Registered Nurses ┆ Public Sector          ┆ 8495  │
│ 2006 ┆ Registered Nurses ┆ Private Sector         ┆ 4566  │
│ 2006 ┆ Registered Nurses ┆ Not in Active Practice ┆ 2391  │
│ 2006 ┆ Enrolled Nurses   ┆ Public Sector          ┆ 2956  │
│ 2006 ┆ Enrolled Nurses   ┆ Private Sector         ┆ 1484  │
└──────┴───────────────────┴────────────────────────┴───────┘
shape: (9, 5)
┌────────────┬──────────┬───────────────────┬────────────────────────┬─────────────┐
│ statistic  ┆ year     ┆ type              ┆ sector                 ┆ count       │
│ ---        ┆

In [55]:
# check if there are same columns
print(datasets["workforce_doctors"].columns, datasets["workforce_nurses"].columns, datasets["workforce_pharmacists"].columns)
print(datasets["capacity_hospital_beds"].columns, datasets["capacity_primary_care"].columns)

['year', 'sector', 'specialist_non-specialist', 'count'] ['year', 'type', 'sector', 'count'] ['year', 'sector', 'count']
['year', 'institution_type', 'facility_type_a', 'public_private', 'no_of_facilities', 'no_beds'] ['year', 'institution_type', 'sector', 'facility_type_b', 'no_of_facilities']


In [56]:
# merge related datasets
all_workforce = pl.concat([datasets["workforce_doctors"], datasets["workforce_nurses"], datasets["workforce_pharmacists"]], how="diagonal")

all_capacity = pl.concat([datasets["capacity_hospital_beds"], datasets["capacity_primary_care"]], how="diagonal")

In [ ]:
# fillna , that arises from inconsistent columns in the three datasets
print(all_workforce.null_count())
df_no_na = all_workforce.with_columns(
    pl.col("specialist_non-specialist").fill_null(pl.lit("")),
    pl.col("type").fill_null(pl.lit(""))
)
print(df_no_na.null_count())
describe(df_no_na)

shape: (1, 5)
┌──────┬────────┬───────────────────────────┬───────┬──────┐
│ year ┆ sector ┆ specialist_non-specialist ┆ count ┆ type │
│ ---  ┆ ---    ┆ ---                       ┆ ---   ┆ ---  │
│ u32  ┆ u32    ┆ u32                       ┆ u32   ┆ u32  │
╞══════╪════════╪═══════════════════════════╪═══════╪══════╡
│ 0    ┆ 0      ┆ 168                       ┆ 0     ┆ 120  │
└──────┴────────┴───────────────────────────┴───────┴──────┘
shape: (1, 5)
┌──────┬────────┬───────────────────────────┬───────┬──────┐
│ year ┆ sector ┆ specialist_non-specialist ┆ count ┆ type │
│ ---  ┆ ---    ┆ ---                       ┆ ---   ┆ ---  │
│ u32  ┆ u32    ┆ u32                       ┆ u32   ┆ u32  │
╞══════╪════════╪═══════════════════════════╪═══════╪══════╡
│ 0    ┆ 0      ┆ 0                         ┆ 0     ┆ 0    │
└──────┴────────┴───────────────────────────┴───────┴──────┘
shape: (9, 6)
┌────────────┬─────────────┬───────────────┬────────────────────┬─────────────┬────────────────────┐
│ s

In [60]:
print(all_capacity.null_count())
all_capacity_no_null = all_capacity.with_columns(
    pl.col(pl.String).fill_null(pl.lit("")),
    pl.col("no_beds").fill_null(pl.lit(0))
)
print(all_capacity_no_null.null_count())
describe(all_capacity_no_null)

shape: (1, 8)
┌──────┬──────────────┬──────────────┬──────────────┬─────────────┬─────────┬────────┬─────────────┐
│ year ┆ institution_ ┆ facility_typ ┆ public_priva ┆ no_of_facil ┆ no_beds ┆ sector ┆ facility_ty │
│ ---  ┆ type         ┆ e_a          ┆ te           ┆ ities       ┆ ---     ┆ ---    ┆ pe_b        │
│ u32  ┆ ---          ┆ ---          ┆ ---          ┆ ---         ┆ u32     ┆ u32    ┆ ---         │
│      ┆ u32          ┆ u32          ┆ u32          ┆ u32         ┆         ┆        ┆ u32         │
╞══════╪══════════════╪══════════════╪══════════════╪═════════════╪═════════╪════════╪═════════════╡
│ 0    ┆ 0            ┆ 96           ┆ 96           ┆ 0           ┆ 96      ┆ 180    ┆ 180         │
└──────┴──────────────┴──────────────┴──────────────┴─────────────┴─────────┴────────┴─────────────┘
shape: (1, 8)
┌──────┬──────────────┬──────────────┬──────────────┬─────────────┬─────────┬────────┬─────────────┐
│ year ┆ institution_ ┆ facility_typ ┆ public_priva ┆ no_of_fac

In [63]:
all_capacity_no_null.write_csv("../datasets/all_capacity_cleaned.csv")
df_no_na.write_csv("../datasets/all_workforce_cleaned.csv")

# Summary
- Datasets do not have duplicated rows
- null values are filled with empty strings or 0
- all capacity related data are merged
- all workforce related data are merged